# AlphaGenome profiling: Block 1 (trace -> stage map)
Run top to bottom. Weights go to the VM disk (`/content/ag_data`), results to `experiments/results/`; the last cell carries the results back into this notebook.

To keep weights across sessions, run `%env AG_DATA=/content/drive/MyDrive/alphagenome` (after mounting Drive in the Colab website) before the weights cell.

## 1. Setup: clone/pull the branch and install

In [ ]:
!git clone -b experiments-profiling https://github.com/Shivaritha22/alphagenome-pytorch /content/ag 2>/dev/null || (git -C /content/ag fetch origin && git -C /content/ag checkout -f experiments-profiling && git -C /content/ag reset --hard origin/experiments-profiling)
%cd /content/ag
!pip install -q -e .

remote: Enumerating objects: 16, done.
remote: Counting objects: 100% (16/16), done.
remote: Compressing objects: 100% (6/6), done.
remote: Total 10 (delta 4), reused 10 (delta 4), pack-reused 0 (from 0)
Unpacking objects: 100% (10/10), 1.10 MiB | 5.78 MiB/s, done.
From https://github.com/Shivaritha22/alphagenome-pytorch
   6fdc24c..a74d666  experiments-profiling -> origin/experiments-profiling
M	experiments/results/profile_16kb.txt
M	experiments/results/tree_16kb.txt
Already on 'experiments-profiling'
Your branch is behind 'origin/experiments-profiling' by 1 commit, and can be fast-forwarded.
  (use "git pull" to update your local branch)
Updating 6fdc24c..a74d666
error: Your local changes to the following files would be overwritten by merge:
	experiments/results/profile_16kb.txt
	experiments/results/tree_16kb.txt
Please commit your changes or stash them before you merge.
error: The following untracked working tree files would be overwritten by merge:
	experiments/results/perfetto_16k

## 2. Weights (download once per session)

In [2]:
!python experiments/01_cache_weights.py


model_all_folds.safetensors: downloading bytes:  21% 194M/921M [00:02<00:02, 259MB/s, 13.3MB/s  ]  
model_all_folds.safetensors: downloading bytes:  32% 293M/921M [00:03<00:02, 252MB/s, 24.1MB/s  ]
model_all_folds.safetensors: downloading bytes:  85% 782M/921M [00:09<00:02, 63.6MB/s, 48.3MB/s  ]]
model_all_folds.safetensors: downloading bytes:  86% 789M/921M [00:09<00:02, 64.7MB/s, 48.3MB/s  ] ]
model_all_folds.safetensors: downloading bytes:  87% 797M/921M [00:09<00:02, 55.0MB/s, 48.4MB/s  ] ]
model_all_folds.safetensors: downloading bytes:  88% 813M/921M [00:10<00:03, 29.6MB/s, 46.9MB/s  ]] 
model_all_folds.safetensors: downloading bytes: 100% 813M/813M [00:10<00:00, 74.8MB/s, 45.8MB/s  ]]
model_all_folds.safetensors: reconstructing file: 100% 921M/921M [00:10<00:00, 84.7MB/s, 61.5MB/s  ]
saved: /content/ag_data/weights/model_all_folds.safetensors (921 MB)


## 3. Trace one 16 kb forward pass (sanity run + full-depth capture)

In [3]:
!python experiments/02_trace_json.py --length 16384

sanity 16kb: 21 output tensors, non-finite: none
  atac.1                                        float32[1, 16384, 256]
  atac.128                                      float32[1, 128, 256]
  dnase.1                                       float32[1, 16384, 384]
  dnase.128                                     float32[1, 128, 384]
  procap.1                                      float32[1, 16384, 128]
  procap.128                                    float32[1, 128, 128]
  cage.1                                        float32[1, 16384, 640]
  cage.128                                      float32[1, 128, 640]
  rna_seq.1                                     float32[1, 16384, 768]
  rna_seq.128                                   float32[1, 128, 768]
  chip_tf.128                                   float32[1, 128, 1664]
  chip_histone.128                              float32[1, 128, 1152]
  contact_maps                                  float32[1, 8, 8, 28]
  splice_sites.logits                     

## 4. Top-level view of the tree

In [4]:
!python experiments/03_trace_tree.py --depth 2

# tree_16kb.json  meta={"length": 16384, "device": "cuda", "dtype": "fp32", "python": "3.13.15", "torch": "2.11.0+cu130", "cuda": "13.0", "gpu": "NVIDIA A100-SXM4-80GB", "port_commit": "6fdc24cae123f0c6ea332528fa8170770fea1fd9", "upstream_base": "72268c0", "weights": "model_all_folds.safetensors", "driver": "580.82.07"}
<root> [AlphaGenome]  params=450.45M  in=[[float32[1, 16384, 4],int64[1]],{}] -> out={atac:{1:float32[1, 16384, 256],128:float32[1, 128, 256]},dnase:{1:...  [glue]
  encoder [SequenceEncoder]  params=89.97M  in=[[float32[1, 16384, 4]],{}] -> out=[float32[1, 1536, 128],{bin_size_1:float32[1, 768, 16384],bin_size_...  [S1_encoder]
    encoder.dna_embedder [DnaEmbedder]  params=3.00M  in=[[float32[1, 4, 16384]],{}] -> out=float32[1, 768, 16384]  [S1_encoder]
    encoder.pool [Pool1d] call 1/7  params=0  in=[[float32[1, 768, 16384]],{}] -> out=float32[1, 768, 8192]  [S1_encoder]
    encoder.down_blocks.0 [DownResBlock]  params=7.46M  in=[[float32[1, 768, 8192]],{}] -> out=f

## 5. Full tree + stage table + checks (writes `tree_16kb.txt`)
Check the end of the output: `uncovered top-level modules: none` and `pair_update ... even-only: True`.

In [5]:
!python experiments/03_trace_tree.py --full --ops --stages --save

# tree_16kb.json  meta={"length": 16384, "device": "cuda", "dtype": "fp32", "python": "3.13.15", "torch": "2.11.0+cu130", "cuda": "13.0", "gpu": "NVIDIA A100-SXM4-80GB", "port_commit": "6fdc24cae123f0c6ea332528fa8170770fea1fd9", "upstream_base": "72268c0", "weights": "model_all_folds.safetensors", "driver": "580.82.07"}
<root> [AlphaGenome]  params=450.45M  in=[[float32[1, 16384, 4],int64[1]],{}] -> out={atac:{1:float32[1, 16384, 256],128:float32[1, 128, 256]},dnase:{1:...  [glue]
  encoder [SequenceEncoder]  params=89.97M  in=[[float32[1, 16384, 4]],{}] -> out=[float32[1, 1536, 128],{bin_size_1:float32[1, 768, 16384],bin_size_...  [S1_encoder]
    . Tensor.transpose  [float32[1, 16384, 4]] -> float32[1, 4, 16384]
    encoder.dna_embedder [DnaEmbedder]  params=3.00M  in=[[float32[1, 4, 16384]],{}] -> out=float32[1, 768, 16384]  [S1_encoder]
      encoder.dna_embedder.conv1 [Conv1d]  params=46.8K  in=[[float32[1, 4, 16384]],{}] -> out=float32[1, 768, 16384]  [S1_encoder]
        . torch

## 6. Per-op profile (writes `profile_16kb.txt`, `perfetto_16kb.json`)

In [6]:
!python experiments/04_profile_ops.py --length 16384

/usr/local/lib/python3.13/dist-packages/torch/profiler/profiler.py:224: UserWarning: Warning: Profiler clears events at the end of each cycle.Only events from the current cycle will be reported.To keep events across cycles, set acc_events=True.
  _warn_once(
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg     Self CUDA   Self CUDA %    CUDA total  CUDA time avg    # of Calls  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                         S2_tower:tower         0.00%       0.000us         0.00%       0.000us       0.000us      67.263ms       

## 7. Graph capture, two levels (whole model)
`dynamo` = Python-level ops; `aten` = core-ATen primitives (what a compiler backend receives; metadata-only ops like view/permute are drawn grey). Check each first line: want `1 graph(s), 0 graph break(s)`, `0 unresolved module paths`.

In [ ]:
!python experiments/06_dynamo_graph.py --length 16384 --level dynamo
!python experiments/06_dynamo_graph.py --length 16384 --level aten

## 8. Zoom: op-level graph of one module (reuses the saved capture)
Change `--root` to any module, e.g. `tower.blocks.0.pair_update`, `tower.blocks.1.mha`, `encoder.down_blocks.0`; `--level dynamo` for the Python-level view. Refuses if the module has more than `--max-nodes` ops; pick a deeper one.

In [ ]:
!python experiments/06_dynamo_graph.py --length 16384 --level aten --render-only --root tower.blocks.0.mha --depth 1 --ops

## 9. Export results
Prints the results zip into this cell's output. Save the notebook afterwards, then locally run:
`python experiments/export_results.py unpack experiments/run.ipynb`

In [7]:
!python experiments/export_results.py export

4 files, zip 0.55 MB, base64 0.73 MB
  perfetto_16kb.json  (7419 KB)
  profile_16kb.txt  (13 KB)
  tree_16kb.json  (462 KB)
  tree_16kb.txt  (297 KB)
=====BEGIN_AG_RESULTS_ZIP=====
...
AAAAAAQABAD1AAAAuWIIAAAA
=====END_AG_RESULTS_ZIP=====
